# ShadeAI — Train, Test, Evaluate, and Save a Model

This notebook trains a `RandomForestRegressor`, evaluates it on a held-out test set, and saves the full preprocessing/model pipeline as a `.pkl` file for later use.

**Dataset expected:** `shadeai_mixed_synthetic_real_dataset.csv` in the same folder as this notebook.

> **Important:** In the current dataset, `shade_fraction` is a synthetic proxy label, not measured shade. Metrics in this notebook assess prediction of those synthetic labels only, not real-world shade accuracy.

## 1. Install/import dependencies
Run the install command in a terminal if a package is missing: `pip install pandas numpy scikit-learn joblib matplotlib`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_PATH = Path("shadeai_mixed_synthetic_real_dataset.csv")
MODEL_PATH = Path("models/shadeai_model.pkl")

print("Dataset path:", DATA_PATH.resolve())
print("Dataset exists:", DATA_PATH.exists())


: 

## 2. Load and inspect the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Put the CSV beside this notebook or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))


## 3. Choose target and input features
The target is `shade_fraction`. We deliberately exclude `tree_shade_potential` and `synthetic_urban_shadow_prior`, which were used to create the synthetic target; including them would leak the target-generation formula into training and inflate evaluation results. The dataset does not yet contain real building/canopy coverage or observed shade labels.

In [ ]:
TARGET = "shade_fraction"

numeric_features = [
    "tree_count", "mean_dbh_cm", "median_dbh_cm", "total_dbh_cm",
    "healthy_tree_count", "fair_tree_count", "poor_tree_count",
    "species_count", "trees_per_hectare", "healthy_tree_ratio",
    "log_tree_count", "log_total_dbh_cm", "non_good_tree_count",
    "non_good_tree_ratio", "dbh_per_tree_cm",
    "center_lat", "center_lon", "latitude_rad", "longitude_rad",
    "hour", "day_of_year",
]
categorical_features = ["borough", "scenario"]
features = numeric_features + categorical_features

required = [TARGET] + features
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()
df = df[df[TARGET].between(0, 1)].copy()

X = df[features].copy()
y = df[TARGET].astype(float)

print("Rows with labels:", len(df))
print("Input features:", len(features))
print("Missing target labels:", int(df[TARGET].isna().sum()))
if "label_type" in df.columns:
    print("Label types:", df["label_type"].value_counts(dropna=False).to_dict())


## 4. Split into training and test sets
The same grid cell appears under multiple time scenarios. Grouping by `grid_x` and `grid_y` keeps a cell's scenarios together, reducing leakage between train and test sets.

In [ ]:
if {"grid_x", "grid_y"}.issubset(df.columns):
    groups = df["grid_x"].astype(str) + "_" + df["grid_y"].astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))
else:
    # Fallback if grid identifiers are absent.
    from sklearn.model_selection import train_test_split
    train_idx, test_idx = train_test_split(
        np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE
    )

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows:  {len(X_test):,}")


## 5. Build and train the model

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
print("Training complete.")


## 6. Test and evaluate

In [ ]:
predictions = np.clip(model.predict(X_test), 0.0, 1.0)

mae = mean_absolute_error(y_test, predictions)
rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
r2 = r2_score(y_test, predictions)

metrics = {
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "MAE": float(mae),
    "RMSE": rmse,
    "R2": float(r2),
    "target": TARGET,
    "label_type": (
        str(df["label_type"].iloc[0]) if "label_type" in df.columns else "unknown"
    ),
    "interpretation": "Metrics evaluate synthetic proxy labels only; not real shade accuracy.",
}
print(json.dumps(metrics, indent=2))


## 7. Plot actual-vs-predicted target values

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predictions, alpha=0.35)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Synthetic target shade_fraction")
plt.ylabel("Predicted shade_fraction")
plt.title("Test set: predictions vs synthetic labels")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(True, alpha=0.25)
plt.show()


## 8. Save the trained pipeline as a `.pkl` file
The file includes preprocessing and the estimator, so later inference uses the same feature transformations.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
artifact = {
    "model": model,
    "feature_columns": features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target": TARGET,
    "metrics": metrics,
    "label_warning": (
        "Trained on synthetic shade_fraction labels; not validated for real-world shade."
    ),
}
joblib.dump(artifact, MODEL_PATH)
print("Saved model:", MODEL_PATH.resolve())
print("File size (bytes):", MODEL_PATH.stat().st_size)


## 9. Reload the `.pkl` and test it

In [ ]:
loaded_artifact = joblib.load(MODEL_PATH)
loaded_model = loaded_artifact["model"]
sample = X_test.head(10).copy()
sample_predictions = np.clip(loaded_model.predict(sample), 0, 1)

results = sample.copy()
results["actual_synthetic_shade_fraction"] = y_test.loc[sample.index]
results["predicted_shade_fraction"] = sample_predictions
display(results[[
    "borough", "scenario", "hour", "tree_count",
    "actual_synthetic_shade_fraction", "predicted_shade_fraction"
]])


## 10. Use the saved model in another Python file

```python
import joblib
import pandas as pd

artifact = joblib.load("models/shadeai_model.pkl")
model = artifact["model"]
feature_columns = artifact["feature_columns"]

# new_data must contain all the same input columns as training.
new_data = pd.read_csv("new_route_features.csv")
predicted_shade_fraction = model.predict(new_data[feature_columns]).clip(0, 1)
print(predicted_shade_fraction)
```

Use the same versions of scikit-learn and related packages when loading a pickle. Only load `.pkl` files you trust.

**Before claiming ShadeAI predicts real shade:** replace the synthetic `shade_fraction` target with measured or physically simulated shade labels and retrain.